# 第4回　Python統計処理入門
## ―― 計算は機械に、判断は人間に

統計学Ⅰ（B）　／　北星学園大学

### 今日から、あなたがコードを書く

ここまで電卓やグラフ用紙でやってきた人、お疲れさま。**これからは1秒だ。**

第2回（代表値）・第3回（ばらつき）でやったことを、今日は**自分のコードで**求める。といっても全部書くわけではない。コードの大半は用意してある。あなたがやるのは、空欄 **`____`** を埋めることだけ。

> ルール：セルを上から実行し、**`____` を正しい言葉に書き換えてから** ▶ を押す。
> エラーが出ても大丈夫。エラーは「ここが違うよ」という**お知らせ**であって、失敗ではない。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 1. データは「表」＝ DataFrame

読み込んだデータ `df` は、Excelのような**表**だ。Pythonではこれを **DataFrame** と呼ぶ。まず大きさと列名を見てみよう。（このセルはそのまま実行）

In [ ]:
print("行数・列数:", df.shape)        # (種数, 列数)
print("列名:", list(df.columns))     # どんな列があるか
df.head()                            # 先頭5行を表示

---
## 2. 列を1つ取り出す　🔧穴埋め①

表から1列を取り出すには `df["列名"]` と書く。上のセルで見た列名から **`体重g`** を取り出そう。

下の `____` を **体重g** に書き換えて実行。

In [ ]:
# ____ に列名を入れる（前のセルの列名を見てね）
df["____"].head()

---
## 3. 代表値をコードで（第2回の復習）　🔧穴埋め②③

第2回で見た平均・中央値を、今度はコードで。

- 平均は `.mean()`　← これは見本
- 中央値は `.median()`、最頻値は `.mode()`

`____` に正しいメソッド名を入れよう。

**第2回の数字（平均5,881g・中央値3,006g）と一致するか確かめること。**

In [ ]:
print("平均　", df["体重g"].mean())        # 見本
print("中央値", df["体重g"].____())       # ヒント: median
print("最頻値", df["体重g"].____().iloc[0])   # ヒント: mode

> **最頻値に注意。** 体重は連続した数値なので、まったく同じ値の種はほぼいない。出てきた「最頻値」は、たまたま一致した1つの値にすぎず、**意味がない**。
> 最頻値が役に立つのは、`一腹産子数` のように**とびとびの値**をとる列である。試してみよう。

In [ ]:
print("一腹産子数の最頻値:", df["一腹産子数"].mode().iloc[0])
print("\n値ごとの種数（上位5）:")
print(df["一腹産子数"].round(1).value_counts().head(5))

---
## 4. ばらつきをコードで（第3回の復習）　🔧穴埋め④

標準偏差は `.std()`、分散は `.var()`。
`____` に **std** を入れて、
第3回で見た `妊娠期間日` のSD（約37.8日）と一致するか確かめよう。

In [ ]:
print("妊娠期間日  標準偏差", df["妊娠期間日"].____().round(1), "日")
print("妊娠期間日  分散　　", df["妊娠期間日"].var().round(1))
print()
print("体重g　　　 標準偏差", df["体重g"].std().round(0), "g")
print("体重g　　　 平均　　", df["体重g"].mean().round(0), "g")

**体重のSD（約13,135g）は、平均（約5,881g）より大きい。**

ばらつきが平均を上回るのは、**分布が極端に歪んでいる合図**である（第2・3回でやったとおり）。こういう列を見たら、平均だけで語ってはいけない。

---
## 5. describe() ―― 一気に要約

`.describe()` を使うと、件数・平均・SD・最小最大・四分位数をまとめて出せる。手計算なら何十分もかかる作業が一瞬。（そのまま実行）

**最小値と最大値を必ず見ること。** ありえない値が入っていないかの確認である。

In [ ]:
df[["体重g", "妊娠期間日", "集団サイズ", "最長寿命月"]].describe().round(1)

---
## 6. グループごとに集計　🔧穴埋め⑤

「**科ごと**の体重の平均」を出す。`df.groupby("グループにする列")["集計する列"].mean()` と書く。
`____` に **科** を入れよう。
どの科が一番大きい？

In [ ]:
# ____ にグループにしたい列名（科）を入れる
df.groupby("____")["体重g"].mean().round(0).sort_values(ascending=False)

---
## 7. 欠損を確認（そのまま実行）

第6回で扱う「どの項目が測られていないか」が、コードだと一瞬で分かる。

In [ ]:
欠損 = df.isna().sum()
print("列ごとの欠損数（多い順）:")
print(欠損.sort_values(ascending=False).head(8))
print(f"\n全 {len(df)} 種中、体重の記録がある種: {df['体重g'].notna().sum()} 種")

**本物の研究データは、こういう姿をしている。** 体重ですら111種が未測定である。
第6回で「この偏りが何を意味するか」を詳しく扱う。

---
## 8. グラフを1行で　🔧穴埋め⑥

第2・3回で見たヒストグラムも、コードなら1行。`____` に **体重g** を入れよう。

In [ ]:
plt.figure(figsize=(7,3.5))
plt.hist(df["____"].dropna(), bins=40, color="#80cbc4", edgecolor="white")
plt.xlabel("体重（g）"); plt.ylabel("種数"); plt.title("ヒストグラム（手作業ゼロ）")
plt.show()

右に長い裾。第2回で見たとおりの形が、1行で出せた。

**対数の目盛りにするのも1行。** 第3回でやった変換を自分でやってみよう。

In [ ]:
plt.figure(figsize=(7,3.5))
plt.hist(np.log10(df["体重g"].dropna()), bins=40, color="#80cbc4", edgecolor="white")
plt.xticks([1,2,3,4,5], ["10g","100g","1kg","10kg","100kg"])
plt.xlabel("体重（対数目盛り）"); plt.ylabel("種数"); plt.title("対数にすると山が見える")
plt.show()

---
## 今日のまとめ

- データ＝**表(DataFrame)**。`df["列名"]` で列を取り出す。
- 代表値 `.mean() .median() .mode()`、ばらつき `.std() .var()`、要約 `.describe()`。
- グループ集計 `df.groupby("列")["列"].mean()`、欠損 `df.isna().sum()`、グラフ `plt.hist(...)`。

第2・3回で見た値と、今日**コードで**求めた値は同じだったはず。違いは速さと正確さ ―― **計算は機械に任せ、人間は『どの数字を信じるか』の判断に集中する。**
次回からはこの道具の上に、相関・推定・検定…を積み上げていく。

> 🔧 **困ったとき（よくあるエラー）**
> - `KeyError: '____'` → 列名のスペル違い。前のセルの `列名:` と見比べる。
> - `SyntaxError` / インデント → 半角・全角、カッコの対応を確認。
> - 空欄が埋まっていない → `____` が残っていないか探す。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。